In [1]:
import pandas as pd
from catboost import CatBoostRegressor

In [2]:
dev_df = pd.read_csv("train-test.csv")
final_df = pd.read_csv("validation.csv")
december_df = pd.read_csv("december-chart-inputs.csv")

In [3]:
dev_df["date"] = pd.to_datetime(dev_df["date"])
final_df["date"] = pd.to_datetime(final_df["date"])
december_df["date"] = pd.to_datetime(december_df["date"])

In [4]:
dev_df["weight"] = dev_df["weight"].abs()
final_df["weight"] = final_df["weight"].abs()
december_df["weight"] = december_df["weight"].abs()

weight_median = dev_df["weight"].median()

dev_df["weight"] = dev_df["weight"].fillna(weight_median)
final_df["weight"] = final_df["weight"].fillna(weight_median)
december_df["weight"] = december_df["weight"].fillna(weight_median)

In [5]:
dev_daily_market = (
    dev_df.groupby("date")["market_index"]
    .transform("median")
)

dev_df["market_index"] = (
    dev_df["market_index"]
    .fillna(dev_daily_market)
)

final_daily_market = (
    final_df.groupby("date")["market_index"]
    .transform("median")
)

final_df["market_index"] = (
    final_df["market_index"]
    .fillna(final_daily_market)
)

market_fallback = dev_df["market_index"].median()

dev_df["market_index"] = dev_df["market_index"].fillna(market_fallback)
final_df["market_index"] = final_df["market_index"].fillna(market_fallback)

In [6]:
def add_features(frame):
    frame = frame.copy()

    frame["month"] = frame["date"].dt.month
    frame["day_of_week_num"] = frame["date"].dt.dayofweek
    frame["route"] = frame["pickup"] + " -> " + frame["delivery"]

    return frame

In [7]:
dev_df = add_features(dev_df)
final_df = add_features(final_df)
december_df = add_features(december_df)

In [8]:
categorical_features = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

numeric_features = [
    "distance",
    "weight",
    "month",
    "day_of_week_num"
]

In [9]:
rich_categorical_features = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

rich_numeric_features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "month",
    "day_of_week_num"
]

rich_features = (
    rich_categorical_features +
    rich_numeric_features
)

In [10]:
print("Development shape:", dev_df.shape)
print("Validation shape:", final_df.shape)
print("December shape:", december_df.shape)

print("\nDevelopment missing values:")
print(dev_df[rich_features + ["posted_rate"]].isna().sum())

print("\nValidation missing values:")
print(final_df[rich_features].isna().sum())

print("\nDecember missing values:")
print(
    december_df[
        categorical_features + numeric_features
    ].isna().sum()
)

Development shape: (48000, 17)
Validation shape: (12000, 16)
December shape: (31, 10)

Development missing values:
pickup             0
delivery           0
equipment          0
route              0
pickup_lat         0
pickup_lon         0
delivery_lat       0
delivery_lon       0
distance           0
weight             0
market_index       0
quote_signal       0
month              0
day_of_week_num    0
posted_rate        0
dtype: int64

Validation missing values:
pickup             0
delivery           0
equipment          0
route              0
pickup_lat         0
pickup_lon         0
delivery_lat       0
delivery_lon       0
distance           0
weight             0
market_index       0
quote_signal       0
month              0
day_of_week_num    0
dtype: int64

December missing values:
pickup             0
delivery           0
equipment          0
route              0
distance           0
weight             0
month              0
day_of_week_num    0
dtype: int64


In [11]:
final_rich_model = CatBoostRegressor(
    iterations=220,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

X_dev_rich = dev_df[rich_features]
y_dev = dev_df["posted_rate"]

X_final_rich = final_df[rich_features]

final_rich_model.fit(
    X_dev_rich,
    y_dev,
    cat_features=rich_categorical_features
)

final_predictions = final_rich_model.predict(X_final_rich)

0:	learn: 1427.8960669	total: 89.1ms	remaining: 19.5s
100:	learn: 586.0444954	total: 3.83s	remaining: 4.51s
200:	learn: 576.4646731	total: 7.46s	remaining: 705ms
219:	learn: 575.0813544	total: 8.11s	remaining: 0us


In [12]:
print("Predictions:", len(final_predictions))
print("Min:", final_predictions.min())
print("Max:", final_predictions.max())
print("Mean:", final_predictions.mean())
print("Non-positive:", (final_predictions <= 0).sum())

Predictions: 12000
Min: 257.7218503579488
Max: 6844.111807246676
Mean: 2393.234663572118
Non-positive: 0


In [13]:
template = pd.read_csv("validation-predictions-template.csv")

prediction_map = dict(
    zip(final_df["load_id"], final_predictions)
)

template["predicted_rate"] = (
    template["load_id"].map(prediction_map)
)

print("Shape:", template.shape)
print("\nMissing values:")
print(template.isna().sum())

print("\nDuplicate IDs:", template["load_id"].duplicated().sum())
print("Non-positive predictions:", (template["predicted_rate"] <= 0).sum())

template.head()

Shape: (12000, 2)

Missing values:
load_id           0
predicted_rate    0
dtype: int64

Duplicate IDs: 0
Non-positive predictions: 0


,load_id,predicted_rate
0,TE-000001,861.250522
1,TE-000002,5051.793995
2,TE-000003,5313.958336
3,TE-000004,4087.334492
4,TE-000005,1838.183325


In [14]:
template.to_csv("validation_predictions.csv",index=False)
print("Saved validation_predictions.csv")

Saved validation_predictions.csv


In [15]:
final_common_model = CatBoostRegressor(
    iterations=230,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

X_dev_common = dev_df[
    categorical_features + numeric_features
]

X_december = december_df[
    categorical_features + numeric_features
]

final_common_model.fit(
    X_dev_common,
    y_dev,
    cat_features=categorical_features
)

december_predictions = final_common_model.predict(
    X_december
)

0:	learn: 1427.5348798	total: 37.3ms	remaining: 8.53s
100:	learn: 590.3742879	total: 3.56s	remaining: 4.55s
200:	learn: 582.3289711	total: 7.01s	remaining: 1.01s
229:	learn: 580.3972052	total: 8s	remaining: 0us


In [16]:
print("December predictions:", len(december_predictions))
print("Min:", december_predictions.min())
print("Max:", december_predictions.max())
print("Mean:", december_predictions.mean())
print("Non-positive:", (december_predictions <= 0).sum())

December predictions: 31
Min: 868.854268790788
Max: 898.0227613243719
Mean: 885.600632530601
Non-positive: 0


In [17]:
december_submission = december_df[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date"
    ]
].copy()

december_submission["predicted_rate"] = december_predictions

december_submission["date"] = (
    december_submission["date"]
    .dt.strftime("%Y-%m-%d")
)

december_submission.to_csv(
    "december_chart_inputs_completed.csv",
    index=False
)

print("Saved december_chart_inputs_completed.csv")
print("Shape:", december_submission.shape)
print(
    "Non-positive predictions:",
    (december_submission["predicted_rate"] <= 0).sum()
)

december_submission.head()

Saved december_chart_inputs_completed.csv
Shape: (31, 7)
Non-positive predictions: 0


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,868.854269
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,881.816759
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,898.022761
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,896.745990
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,890.451177
